# 5차시 (3) RAG 챗봇 만들기 — 정답

한경국립대 2026 여름특강 · 딥러닝/머신러닝 입문 (초급반)

> **2차시(온라인)** 에서 NotebookLM 같은 **완성된 서비스로** '내 문서에 근거해 답하는' RAG를 체험했죠.
> **오늘(오프라인)** 은 그 RAG를 *서비스로 쓰는 게 아니라* **직접 코드로** 만들어, 문서 적재 → 검색 → 답 생성까지 **한 조각씩 화면으로 확인**합니다.

## RAG가 뭔가요?

앞 노트북에서, 답의 근거가 될 텍스트를 **직접 프롬프트에 붙여넣어** 주면 모델이 더 정확히 답하는 걸 봤습니다.
하지만 문서가 100쪽이라면? 매번 통째로 붙여넣을 수는 없습니다.

**RAG (Retrieval-Augmented Generation, 검색 증강 생성)** 은 질문과 관련된 **부분만 찾아(검색)**, 그 근거로 **답을 생성** 합니다.

## 오늘 만들 RAG 파이프라인 (6단계)

```
[1] 문서 적재   원본 문서를 불러온다
      ↓
[2] 청킹       긴 문서를 작은 조각(chunk)으로 자른다
      ↓
[3] 임베딩     각 조각을 '의미 좌표' 숫자 벡터로 바꾼다
      ↓
[4] 벡터 저장   벡터들을 벡터DB(Chroma)에 넣는다
      ↓
[5] 검색       질문도 벡터로 바꿔, 가장 비슷한 조각 top-k를 찾는다 (코사인 유사도)
      ↓
[6] 생성       찾은 조각을 근거로 LLM이 최종 답을 만든다
```

아래에서 **한 단계씩 셀을 나눠** 직접 만들어 보고, 마지막에 **Gradio 챗봇**으로 합칩니다.

> 3차시에서 배운 **벡터·코사인 유사도** 가 [3]·[5]단계에서 그대로 쓰입니다. "비슷한 의미 = 가까운 벡터".

---
## 0. 환경 세팅

### 0-1. 라이브러리 설치
RAG에 필요한 도구를 설치합니다(1~2분).
- `langchain` 계열: 문서 적재·청킹·검색·LLM 연결을 묶어 주는 프레임워크
- `langchain-chroma` / `chromadb`: 벡터를 저장·검색하는 가벼운 **벡터DB**
- `langchain-openai`: OpenAI 임베딩·LLM 호출

설치 후 빨간 의존성 경고가 떠도 무시해도 됩니다.

> ⚠️ **실행 환경**: 이 노트북의 **[3]임베딩·[4]저장·[5]검색·[6]생성** 셀은 **Colab에서 OpenAI API 키**가 있어야 실행됩니다. (아래 `0-2` 에서 키를 입력) 문서 적재·청킹까지는 키 없이도 됩니다.

In [ ]:
!pip install -qU langchain langchain-openai langchain-community
!pip install -qU langchain-chroma chromadb
!pip install -qU "gradio>=5.0"
print("설치 완료!")

### 0-2. OpenAI API 키 입력
주최측에서 받은 키를 붙여넣고 Enter (화면에 보이지 않습니다).

In [ ]:
import os
import getpass

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API 키를 입력하세요: ")

print("API 키 설정 완료" if os.environ.get("OPENAI_API_KEY") else "키가 비어 있습니다.")

---
## [1] 문서 적재 — 답변의 근거가 될 문서 준비

실습이니 외부 파일 없이, **가상의 사내 안내 문서**를 코드로 직접 만들어 `docs/` 폴더에 `.txt` 로 저장합니다.
(실제로는 PDF·웹페이지·회사 위키 등을 불러옵니다. 형식만 다를 뿐 이후 단계는 동일합니다.)

주제: *한경 테크 신입사원 안내서* — 휴가, 재택, 장비, 식대 같은 사내 규정.

In [ ]:
import os

os.makedirs("docs", exist_ok=True)

documents = {
    "휴가.txt": """[연차 휴가 안내]
신입사원은 입사 첫 해에 11일의 연차가 부여됩니다.
연차는 입사 1년 후부터 매년 15일로 늘어나며, 3년차부터 2년마다 1일씩 추가됩니다.
연차 신청은 최소 3일 전에 그룹웨어에서 신청해야 합니다.
반차(반일 휴가)도 사용할 수 있으며, 오전 반차와 오후 반차로 나뉩니다.""",

    "재택근무.txt": """[재택근무 규정]
한경 테크는 주 2회까지 재택근무가 가능합니다.
재택근무를 하려면 전날 오후 6시까지 팀장에게 신청해야 합니다.
재택근무 시에도 코어타임(오전 10시 ~ 오후 4시)에는 메신저에 접속해 있어야 합니다.
신입사원은 입사 후 3개월(수습 기간)이 지나야 재택근무를 신청할 수 있습니다.""",

    "장비.txt": """[업무 장비 지원]
전 직원에게 노트북 1대가 지급됩니다. 개발 직군은 메모리 32GB 모델을 받습니다.
외부 모니터는 1인당 최대 2대까지 지원됩니다.
키보드와 마우스는 자유롭게 신청할 수 있으며, 신청 후 3일 이내에 지급됩니다.
장비가 고장 나면 IT 헬프데스크(내선 1234)로 문의하면 됩니다.""",

    "식대.txt": """[식대 및 복지]
점심 식대로 매일 1만 원이 지원되며, 사내 식당 또는 제휴 식당에서 사용할 수 있습니다.
야근 시(오후 8시 이후) 저녁 식대 1만 원이 추가로 지원됩니다.
매월 마지막 주 금요일은 '패밀리 데이'로 오후 5시에 조기 퇴근합니다.
사내 카페에서는 모든 음료를 50% 할인된 가격에 구매할 수 있습니다.""",
}

for filename, content in documents.items():
    with open(os.path.join("docs", filename), "w", encoding="utf-8") as f:
        f.write(content)

print("문서 저장 완료:", os.listdir("docs"))

이제 저장한 `.txt` 들을 LangChain의 **로더(loader)** 로 불러옵니다.
`DirectoryLoader` 는 폴더 안 파일들을 한 번에 읽어 `Document` 객체 목록으로 만들어 줍니다.
각 `Document` 는 본문(`page_content`)과 출처 정보(`metadata`)를 가집니다.

In [ ]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader

loader = DirectoryLoader(
    "docs",
    glob="*.txt",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"},
)
raw_docs = loader.load()

print(f"불러온 문서 수: {len(raw_docs)}개\n")
print("첫 문서 출처:", raw_docs[0].metadata)
print("첫 문서 내용 미리보기:\n", raw_docs[0].page_content[:80], "...")

### 📄 (직접) 내 PDF 파일에서 텍스트 뽑아내기

방금은 우리가 만든 `.txt` 를 불러왔지만, **실무 문서는 대부분 PDF** 입니다. 오픈소스 PDF 파서(`pypdf`)를 붙이면 여러분이 올린 PDF에서 **텍스트를 추출** 해 똑같이 RAG에 넣을 수 있어요.


<div style="border:1px solid #E7DCC6;border-left:6px solid #C15F3C;border-radius:10px;background:#FBF4EA;padding:16px 20px;margin:12px 0;">
<div style="font-size:16px;font-weight:700;color:#A8472A;margin-bottom:8px;">🔧 직접 해보기 · 내 PDF 올려 보기 &nbsp;(⚠️ Colab 전용)</div>
<div style="color:#3D3A33;line-height:1.8;font-size:14.5px;">
<b>문제</b> &nbsp; 아래 셀을 실행하면 <b>파일 업로드 창</b>이 뜹니다. 손에 있는 <b>PDF 한 개</b>(강의자료·안내문 등)를 올려, 추출된 페이지 수·미리보기를 확인하세요.<br><b>힌트</b> &nbsp; 파일 업로드는 <code>google.colab.files.upload()</code> 로 동작해 <b>Colab에서만</b> 창이 뜹니다. 없으면 인터넷에서 짧은 PDF를 하나 받아 올려도 됩니다.<br><b>관찰</b> &nbsp; <code>PyPDFLoader</code> 가 만든 <code>pdf_docs</code> 도 <code>raw_docs</code> 와 같은 <code>Document</code> 리스트라, <b>[2]청킹부터 그대로</b> 넣으면 내 PDF로 답하는 RAG가 됩니다.
</div>
</div>

In [ ]:
# ⚠️ Colab 전용 — 실행하면 업로드 창이 뜹니다 (여기서 실행 금지)
try:
    from google.colab import files
    uploaded = files.upload()
    pdf_path = list(uploaded.keys())[0]
except Exception:
    pdf_path = "sample.pdf"   # (로컬 실행이면 PDF 경로를 직접 지정하세요)
print("업로드된 파일:", pdf_path)

In [ ]:
# ⚠️ Colab 전용 (PDF 파서 설치·실행)
!pip install -qU pypdf
from langchain_community.document_loaders import PyPDFLoader

pdf_docs = PyPDFLoader(pdf_path).load()   # 페이지별 Document 리스트로 추출
print(f"PDF 페이지 수: {len(pdf_docs)}")
print("1페이지 미리보기:\n", pdf_docs[0].page_content[:300], "...")

> 이렇게 뽑은 `pdf_docs` 도 `raw_docs` 와 **똑같은 `Document` 리스트** 라, 아래 **[2]청킹부터 그대로** 넣으면 여러분 PDF로 답하는 RAG가 됩니다.
> URL로 바로 받으려면 `PyPDFLoader("https://.../파일.pdf")` 처럼 URL을 넣어도 돼요.
> 본문 실습은 이해를 위해 **짧은 사내 안내문(txt)** 으로 계속 진행하고, 마지막 도전 과제에서 내 문서로 바꿔 봅니다.

---
## [2] 청킹 — 긴 문서를 작은 조각으로 자르기

**왜 자르나요?**
- 검색은 "질문과 관련된 *부분*"을 찾는 것이라, 문서가 통째로 한 덩어리면 관련 없는 내용까지 딸려옵니다.
- 작게 자르면(=chunk) 더 **정확한 부분**만 골라 LLM에 줄 수 있습니다.

`RecursiveCharacterTextSplitter` 를 씁니다.
- `chunk_size` : 한 조각의 최대 글자 수
- `chunk_overlap` : 조각끼리 겹치는 글자 수 (문장이 경계에서 잘려 맥락이 끊기는 걸 줄임)

이번엔 데모를 따라 치는 게 아니라, **여러분이 직접** 이 조각내기 코드를 완성합니다.

<div style="border:1px solid #E7DCC6;border-left:6px solid #C15F3C;border-radius:10px;background:#FBF4EA;padding:16px 20px;margin:12px 0;">
<div style="font-size:16px;font-weight:700;color:#A8472A;margin-bottom:8px;">✏️ 직접 작성 ① · 청킹 코드 완성하기 &nbsp;(가이드형 — 뼈대를 채우세요)</div>
<div style="color:#3D3A33;line-height:1.8;font-size:14.5px;">
<b>문제</b> &nbsp; 아래 뼈대의 빈 곳(<code>____</code>)을 채워, <code>raw_docs</code> 를 작은 조각(chunk) 리스트로 나누는 코드를 완성하세요.<br><b>힌트</b> &nbsp; <code>RecursiveCharacterTextSplitter</code> 는 <code>chunk_size</code>(한 조각 최대 글자 수)·<code>chunk_overlap</code>(겹침)을 인자로 받습니다. 우리 문서는 짧으니 값은 <b>여러분이 직접</b> 정하세요(실무 문서는 보통 500~1000자). 만든 splitter 의 <code>split_documents(...)</code> 에 <code>raw_docs</code> 를 넘기면 조각 리스트가 나옵니다.<br><b>예상 동작</b> &nbsp; 원본 4개 문서가 여러 개(보통 6~12개) 조각으로 나뉘고, 첫 조각 3개가 출력됩니다.<br><b>덱 연결</b> &nbsp; RAG 6단계 중 <b>[2] 청킹</b> 을 직접 구현합니다.
</div>
</div>

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=150,     # 한 조각 최대 150자
    chunk_overlap=20,   # 앞뒤 20자씩 겹치기
)
chunks = splitter.split_documents(raw_docs)

# --- 확인용 ---
print(f"원본 {len(raw_docs)}개 문서 -> {len(chunks)}개 조각으로 분할\n")
for i, c in enumerate(chunks[:3]):
    print(f"--- 조각 {i} (출처: {os.path.basename(c.metadata['source'])}) ---")
    print(c.page_content)
    print()

<div style="border:1px solid #E7DCC6;border-left:6px solid #C15F3C;border-radius:10px;background:#FBF4EA;padding:16px 20px;margin:12px 0;">
<div style="font-size:16px;font-weight:700;color:#A8472A;margin-bottom:8px;">🔍 만든 뒤 관찰 · chunk_size 를 바꾸면? &nbsp;(값만 바꿔 실행)</div>
<div style="color:#3D3A33;line-height:1.8;font-size:14.5px;">
방금 청킹을 직접 만들었으니, 이제 <code>[80, 300]</code> 에 다른 값(예: <code>40</code>, <code>500</code>)을 넣어 <b>조각 수</b>와 <b>첫 조각 길이</b>가 어떻게 달라지는지 관찰하세요. 값이 작을수록 조각 수↑·길이↓(문맥 끊김), 클수록 조각 수↓·길이↑(관련 없는 내용까지 딸려옴).
</div>
</div>

In [ ]:
# 값만 바꿔 실행: 아래 [80, 300] 에 다른 숫자(예: 40, 500)를 넣어 비교해 보세요.
for size in [80, 300]:
    test_chunks = RecursiveCharacterTextSplitter(chunk_size=size, chunk_overlap=10).split_documents(raw_docs)
    print(f"chunk_size={size:>3} → 조각 {len(test_chunks):>2}개, 첫 조각 길이 {len(test_chunks[0].page_content)}자")

---
## [3] 임베딩 — 글자를 '의미 좌표(벡터)'로 바꾸기

컴퓨터는 글자의 *의미* 를 직접 비교하지 못합니다. 그래서 각 조각을 **숫자 벡터**로 바꿉니다(3차시에서 본 그 벡터!).
핵심 성질: **의미가 비슷한 문장 → 벡터도 가까이** 위치합니다.

OpenAI의 `text-embedding-3-small` 모델을 씁니다(저렴하고 빠름).
먼저 임베딩이 실제로 어떤 숫자인지, 그리고 "비슷한 문장은 가깝다"는 성질을 **직접 확인**해 봅시다.

In [ ]:
# ⚠️ Colab에서 실행 (OpenAI API 키 필요) — 여기서부터 API 호출
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# 문장 하나를 벡터로 바꿔 보기
vec = embeddings.embed_query("연차 휴가는 며칠인가요?")
print("벡터 길이(차원 수):", len(vec))
print("앞부분 미리보기:", [round(x, 3) for x in vec[:8]], "...")

### 잠깐 실험: '비슷한 의미 = 가까운 벡터' 확인하기
두 문장의 벡터가 얼마나 가까운지 **코사인 유사도** 로 잽니다(3차시 내용). 값이 **1에 가까울수록 비슷한 의미**, 0에 가까울수록 무관.

'휴가' 질문을 **관련 문장 · 주제 다른 문장 · 완전 무관한 문장** 셋과 각각 비교해, 유사도가 어떻게 벌어지는지 봅시다.

In [ ]:
import numpy as np

def cosine_similarity(a, b):
    a, b = np.array(a), np.array(b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

query_vec = embeddings.embed_query("휴가는 며칠 쓸 수 있나요?")
related   = embeddings.embed_query("신입사원은 첫 해에 연차 11일을 받습니다.")   # 관련 O
other     = embeddings.embed_query("점심 식대로 매일 1만 원이 지원됩니다.")       # 주제 다름
unrelated = embeddings.embed_query("화성은 태양계의 네 번째 행성이다.")           # 완전 무관

print("휴가 질문 vs '연차 11일' :", round(cosine_similarity(query_vec, related), 3),   " (관련 O   → 높음)")
print("휴가 질문 vs '점심 식대' :", round(cosine_similarity(query_vec, other), 3),     " (다른 주제 → 중간)")
print("휴가 질문 vs '화성 행성' :", round(cosine_similarity(query_vec, unrelated), 3), " (완전 무관 → 가장 낮음)")
print("\n=> 관련도가 높을수록 유사도가 큽니다. RAG는 이 값으로 '관련 조각'만 골라냅니다.")

---
## [4] 벡터 저장 — 벡터DB(Chroma)에 모든 조각 넣기

조각이 수천 개면 매번 직접 유사도를 계산하긴 번거롭습니다. **벡터DB** 가 이 일을 대신해 줍니다.
- 조각들의 벡터를 저장해 두고,
- 질문 벡터가 들어오면 **가장 가까운 조각들을 빠르게 찾아** 줍니다.

`Chroma.from_documents` 한 줄이면 [3]임베딩 + [4]저장을 함께 처리합니다(각 조각을 임베딩해서 DB에 넣음).

In [ ]:
# ⚠️ Colab에서 실행 (각 조각을 임베딩해 벡터DB에 저장 — API 호출)
from langchain_chroma import Chroma

# chunks(조각들)를 임베딩해서 Chroma 벡터DB에 저장
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="hankyung_handbook",
)

print("벡터DB 생성 완료! 저장된 조각 수:", vectorstore._collection.count())

---
## [5] 검색 — 질문과 가장 비슷한 조각 top-k 찾기

이제 질문을 던지면 벡터DB가 **가장 관련 있는 조각 k개**를 코사인 유사도로 골라 줍니다(이를 **top-k 검색** 이라 합니다).
`vectorstore` 에는 이 일을 해 주는 **유사도 검색 메서드**가 이미 들어 있습니다.
이번엔 그걸 감싸는 **검색 함수 `search(query, k)` 를 직접** 만들어, 앞으로 계속 재사용합니다.

<div style="border:1px solid #E7DCC6;border-left:6px solid #C15F3C;border-radius:10px;background:#FBF4EA;padding:16px 20px;margin:12px 0;">
<div style="font-size:16px;font-weight:700;color:#A8472A;margin-bottom:8px;">✏️ 직접 작성 ② · 검색 함수 <code>search(query, k)</code> 만들기 &nbsp;(⚠️ Colab 전용)</div>
<div style="color:#3D3A33;line-height:1.8;font-size:14.5px;">
<b>문제</b> &nbsp; 질문 문자열과 개수 <code>k</code> 를 받아, 벡터DB에서 질문과 <b>가장 비슷한 조각 k개</b>를 찾아 <b>리스트로 반환</b>하는 함수 <code>search(query, k)</code> 를 빈 셀에 직접 작성하세요.<br><b>힌트</b> &nbsp; 이미 만들어 둔 <code>vectorstore</code> 에는 질문과 <code>k</code> 를 넘기면 관련 조각(<code>Document</code>) 리스트를 돌려주는 <b>유사도 검색 메서드</b>가 있습니다. 그 결과를 그대로 반환하면 됩니다. 함수 정의(<code>def</code>)와 반환(<code>return</code>)을 직접 쓰세요.<br><b>예상 동작</b> &nbsp; <code>search("재택근무는 몇 번 가능해?", 2)</code> 가 재택근무 문서 조각을 담은 리스트를 돌려주고, 아래 확인용 셀이 그 내용을 출력합니다.<br><b>덱 연결</b> &nbsp; RAG 6단계 중 <b>[5] 검색(코사인 top-k)</b> 을 직접 구현합니다.
</div>
</div>

In [ ]:
# ⚠️ Colab에서 실행 (vectorstore 가 있어야 동작 — API)
def search(query, k=2):
    return vectorstore.similarity_search(query, k=k)

# --- 확인용 ---
for i, doc in enumerate(search("재택근무는 일주일에 몇 번 할 수 있어?", k=2)):
    print(f"[{i+1}] (출처: {os.path.basename(doc.metadata['source'])})")
    print(doc.page_content, "\n")

<div style="border:1px solid #E7DCC6;border-left:6px solid #C15F3C;border-radius:10px;background:#FBF4EA;padding:16px 20px;margin:12px 0;">
<div style="font-size:16px;font-weight:700;color:#A8472A;margin-bottom:8px;">🔍 만든 뒤 관찰 · top-k 는 클수록 좋을까? &nbsp;(값만 바꿔 실행)</div>
<div style="color:#3D3A33;line-height:1.8;font-size:14.5px;">
방금 만든 <code>search()</code> 로, <code>k</code> 를 <code>1, 2, 6</code> 으로 바꿔 가며 <b>검색된 조각의 출처</b>가 어떻게 바뀌는지 보세요. <code>k</code> 가 커지면 질문과 무관한 출처(식대·장비 등)까지 섞여 들어옵니다(노이즈). 보통 <b>k=2~4</b> 가 적당합니다.
</div>
</div>

In [ ]:
# 값만 바꿔 실행: 네가 만든 search() 로 k 를 바꿔 가며 출처 변화를 관찰하세요.
q = "재택근무는 며칠 가능해?"
for k in [1, 2, 6]:
    sources = [os.path.basename(d.metadata["source"]) for d in search(q, k=k)]
    print(f"k={k} → 가져온 조각 출처: {sources}")
# k를 키우면 '재택근무'와 무관한 문서(식대·장비 등)까지 섞여 들어옵니다.

→ `k=1~2` 는 재택근무 조각만, `k=6` 은 **무관한 조각까지** 섞입니다. 그 무관한 내용이 프롬프트에 들어가면 모델이 헷갈릴 수 있어요. **적당한 k(보통 2~4)** 가 좋습니다.

---
## [6] 생성 — 찾은 조각을 근거로 LLM이 답하기

드디어 마지막 단계이자 RAG의 심장입니다. [5]에서 찾은 조각들을 **프롬프트에 근거로 넣고** LLM에게 답을 만들게 합니다.
앞 노트북의 "참고 텍스트 제공" 과 똑같은 원리인데, 그 텍스트를 **검색으로 자동으로 채운다**는 점만 다릅니다.

이 검색→생성 전 과정을 하나로 묶은 함수 **`rag_answer(query)` 를 여러분이 직접** 작성합니다.
핵심: 프롬프트에서 *"아래 참고 자료만 근거로 답하고, 자료에 없으면 모른다고 답해"* 라고 못 박아 **환각을 줄입니다**.

<div style="border:1px solid #E7DCC6;border-left:6px solid #C15F3C;border-radius:10px;background:#FBF4EA;padding:16px 20px;margin:12px 0;">
<div style="font-size:16px;font-weight:700;color:#A8472A;margin-bottom:8px;">✏️ 직접 작성 ③ · RAG 파이프라인 <code>rag_answer(query)</code> 만들기 &nbsp;★핵심 (⚠️ Colab 전용)</div>
<div style="color:#3D3A33;line-height:1.8;font-size:14.5px;">
<b>문제</b> &nbsp; 질문을 받아 <b>(1)</b> 관련 조각을 검색하고 <b>(2)</b> 그 조각들을 하나의 문맥 문자열로 합친 뒤 <b>(3)</b> '이 자료만 근거로 답하고, 없으면 모른다고 답하라'는 프롬프트를 구성해 <b>(4)</b> <code>llm</code> 으로 답을 생성하는 함수 <code>rag_answer(query)</code> 를 빈 셀에 처음부터 작성하세요.<br><b>힌트</b> &nbsp; 검색은 방금 만든 <code>search(query, k)</code> 를 <b>재사용</b>하세요. 여러 조각의 <code>page_content</code> 를 줄바꿈으로 이어 붙이면 문맥이 됩니다. 프롬프트에는 <b>[참고 자료]</b>와 <b>[질문]</b>을 함께 넣고, "자료에 없으면 지어내지 말고 모른다고 답하라"는 규칙을 <b>여러분 문장으로 반드시</b> 넣으세요. <code>llm.invoke(프롬프트).content</code> 가 최종 답 문자열입니다.<br><b>예상 동작</b> &nbsp; 문서에 있는 질문엔 근거대로 답하고, 문서에 없는 질문(예: 주차장)엔 "찾을 수 없습니다"류로 답합니다.<br><b>덱 연결</b> &nbsp; RAG 6단계의 <b>[5]검색 + [6]생성</b> 을 하나의 함수로 조립합니다.
</div>
</div>

In [ ]:
# ⚠️ Colab에서 실행 (OpenAI LLM 호출 — API 키 필요)
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

def rag_answer(query, k=3):
    # 1) 관련 조각 검색 (앞에서 만든 search 재사용)
    docs = search(query, k=k)
    # 2) 조각들을 하나의 문맥 문자열로 합치기
    context = "\n\n".join(d.page_content for d in docs)
    # 3) 검색 결과를 근거로 넣고 '없으면 모른다' 규칙을 못 박은 프롬프트
    prompt = f"""너는 회사 안내 챗봇이야. 아래 [참고 자료]만 근거로 질문에 답해줘.
참고 자료에 없는 내용은 지어내지 말고 "안내 문서에서 찾을 수 없습니다"라고 답해줘.

[참고 자료]
{context}

[질문] {query}
[답변]"""
    # 4) LLM 호출
    return llm.invoke(prompt).content

# --- 확인용 ---
print(rag_answer("재택근무는 일주일에 몇 번 가능해?"))

In [ ]:
# 문서에 있는 질문
print("Q1:", rag_answer("신입사원 연차는 며칠이야?"))
print()
print("Q2:", rag_answer("개발자는 노트북 메모리 몇 GB 받아?"))
print()
# 문서에 없는 질문 -> '찾을 수 없습니다' 가 나와야 정상
print("Q3:", rag_answer("회사 주차장은 몇 층까지 있어?"))

<div style="border:1px solid #E7DCC6;border-left:6px solid #C15F3C;border-radius:10px;background:#FBF4EA;padding:16px 20px;margin:12px 0;">
<div style="font-size:16px;font-weight:700;color:#A8472A;margin-bottom:8px;">🔍 만든 뒤 관찰 · '없으면 모른다' 규칙의 힘 &nbsp;(관찰)</div>
<div style="color:#3D3A33;line-height:1.8;font-size:14.5px;">
아래 셀은 여러분의 <code>rag_answer</code> 와, <b>'없으면 모른다' 규칙을 뺀</b> 버전(<code>rag_answer_no_guard</code>)을 나란히 비교합니다. 규칙을 빼면 문서에 없는 질문(주차장)에도 그럴듯하게 <b>지어냅니다(환각)</b>. 여러분이 프롬프트에 넣은 그 한 줄이 환각을 막는 핵심임을 확인하세요.
</div>
</div>

In [ ]:
# ① top-k 비교: 같은 질문, k만 다르게 (값 바꿔 관찰)
print("[k=1]", rag_answer("재택근무 며칠 가능해?", k=1))
print("[k=5]", rag_answer("재택근무 며칠 가능해?", k=5))

# ② '없으면 모른다' 규칙을 뺀 함수 — 문서에 없는 질문에 지어내는지 관찰
def rag_answer_no_guard(query, k=3):
    docs = search(query, k=k)
    context = "\n\n".join(d.page_content for d in docs)
    prompt = f"아래 참고 자료를 보고 질문에 답해줘.\n\n[참고 자료]\n{context}\n\n[질문] {query}"  # 규칙 없음
    return llm.invoke(prompt).content

print("\n[규칙 없음] 문서에 없는 질문:", rag_answer_no_guard("회사 주차장은 몇 층까지 있어?"))
# → 규칙이 없으면 그럴듯하게 지어낼 수 있습니다. 그래서 [6]의 '없으면 모른다' 한 줄이 중요!

위 Q3 처럼 **문서에 없는 질문**에는 지어내지 않고 "찾을 수 없다"고 답하는지 확인하세요.
이것이 RAG가 일반 챗봇보다 **신뢰할 수 있는** 이유입니다 — 답의 근거가 *내 문서* 에 있습니다.

---
## [통합] RAG 챗봇을 Gradio 화면으로

지금까지 만든 [5]검색 + [6]생성을 **채팅 화면**에 연결합니다.
노트북 1의 챗봇과 구조는 같고, `response` 함수가 `rag_answer` 를 부르는 것만 다릅니다.

In [ ]:
import gradio as gr

def rag_chat(message, history):
    # 대화 기록(history)은 여기서는 단순화를 위해 사용하지 않고,
    # 매 질문마다 문서에서 근거를 새로 검색해 답합니다.
    return rag_answer(message, k=3)

demo = gr.ChatInterface(
    fn=rag_chat,
    type="messages",
    title="한경 테크 사내 안내 챗봇 (RAG)",
    description="신입사원 안내서를 근거로 답하는 챗봇입니다. 휴가·재택·장비·식대를 물어보세요.",
    examples=["연차는 며칠이야?", "재택근무 신청은 언제까지 해야 해?", "야근하면 저녁 식대 나와?"],
)
demo.launch()

In [ ]:
demo.close()

---

<div style="border:1px solid #E7DCC6;border-left:6px solid #C15F3C;border-radius:10px;background:#FBF4EA;padding:16px 20px;margin:12px 0;">
<div style="font-size:16px;font-weight:700;color:#A8472A;margin-bottom:8px;">✏️ 도전 과제 &nbsp;내 문서로 RAG 챗봇 만들기 (직접 작성)</div>
<div style="color:#3D3A33;line-height:1.8;font-size:14.5px;">
<b>문제</b> &nbsp; 아래 <code>my_docs</code> 에 <b>여러분이 잘 아는 주제</b>의 문장을 2~5개 넣고, 이어지는 셀을 실행해 그 문서로 답하는 챗봇을 만들어 질문해 보세요. (예: 게임 공략, 동아리 규칙, 학과 안내, 자취 레시피)<br><b>힌트</b> &nbsp; 각 항목은 <code>"제목": "본문 여러 줄"</code> 형태. [1]~[4]단계(적재→청킹→임베딩→저장)는 아래 셀이 한 번에 처리하니, 여러분은 <b>문서 내용만</b> 바꾸면 됩니다. 먼저 예시대로 실행해 보고, 그 다음 내용을 바꾸세요.<br><b>관찰(성공)</b> &nbsp; 내가 넣은 문서 내용을 물으면 그 근거로 답하고, 안 넣은 내용을 물으면 "찾을 수 없습니다"라고 답하면 성공입니다.
</div>
</div>

In [ ]:
# 아래 my_docs 에 '여러분이 잘 아는 주제' 를 2~5개 넣으세요. (먼저 이 예시대로 실행 → 그 다음 내용만 바꿔보기)
my_docs = {
    "동아리 안내": """우리 동아리는 매주 화요일 저녁 6시에 모입니다.
가입 신청은 학기 초 2주간 받으며, 회비는 학기당 2만원입니다.
정기 활동은 스터디와 프로젝트이고, MT는 매 학기 1회 진행합니다.""",
    "자취 요리": """김치볶음밥은 밥, 김치, 계란, 참기름이면 됩니다.
팬에 김치를 볶다가 밥을 넣고, 계란을 풀어 섞은 뒤 참기름을 두르면 완성입니다.
5분이면 되고, 스팸을 넣으면 더 맛있습니다.""",
}

# --- 아래는 위에서 배운 [1]~[4] 단계를 한 번에 실행하는 코드입니다(그대로 실행) ---
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma

my_documents = [Document(page_content=text, metadata={"source": title})
                for title, text in my_docs.items()]
my_chunks = RecursiveCharacterTextSplitter(chunk_size=150, chunk_overlap=20).split_documents(my_documents)
my_vectorstore = Chroma.from_documents(my_chunks, embedding=embeddings, collection_name="my_custom_docs")
print("나만의 벡터DB 준비 완료! 조각 수:", my_vectorstore._collection.count())

In [ ]:
# 나만의 문서로 답하기 — 위에서 직접 만든 rag_answer 를 그대로 재사용합니다.
import gradio as gr

# search·rag_answer 는 '호출 시점'의 vectorstore 를 참조하므로,
# 이 한 줄로 내 문서 것으로 바꾸면 같은 함수가 내 문서로 답합니다.
vectorstore = my_vectorstore

demo = gr.ChatInterface(
    fn=lambda message, history: rag_answer(message),
    type="messages",
    title="나만의 RAG 챗봇",
    description="내가 넣은 문서를 근거로 답합니다.",
)
demo.launch()

In [ ]:
demo.close()

---
## 마무리

오늘 우리는 RAG 챗봇의 **6단계 파이프라인** 을 처음부터 직접 만들었습니다:
**문서 적재 → 청킹 → 임베딩 → 벡터 저장(Chroma) → 검색(코사인 top-k) → 생성**.

- 핵심 아이디어: **"내 문서에서 관련 부분을 찾아, 그걸 근거로 답한다"** → 환각이 줄고 출처가 분명해집니다.
- 3차시의 **벡터·코사인 유사도** 가 임베딩·검색에서 그대로 쓰였고, PDF 파서로 **내 문서(PDF)** 도 넣을 수 있음을 봤습니다.
- **top-k 는 무작정 크면 안 되고**(노이즈↑), 프롬프트의 **'없으면 모른다' 규칙** 이 환각을 막는 핵심임을 실험으로 확인했습니다.

> 더 나아가려면(개념만): 키워드 검색을 섞는 **하이브리드 검색**, 검색 결과를 다시 정렬하는 **리랭커(re-ranker)** 등으로 품질을 높입니다. 오늘은 가장 기본인 **의미(dense) 검색** 으로 충분합니다.